# Q‑Learning (Model‑Free) vs. Value Iteration (Model‑Based) on Gridworld

_Generated: 2025-10-22T17:54:51.765296Z_

We build a small **Gridworld** MDP and solve it in two ways:

- **Value Iteration (VI)** — classic **dynamic programming** that uses the known model to compute $V^*$ and the greedy optimal policy $\pi^*$.
- **Q‑Learning (QL)** — **model‑free** temporal‑difference method that learns action‑values $Q(s,a)$ by interacting with the environment using $\epsilon$‑greedy exploration.

We compare the policies and values, plot learning curves, and render trajectories.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports & Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

ACTIONS = np.array([[-1,0],[0,1],[1,0],[0,-1]])  # U,R,D,L
A_NAMES = np.array(['↑','→','↓','←'])
def clamp(x, lo, hi): return max(lo, min(hi, x))

## 2) Gridworld MDP

A rectangular grid with **walls** and **terminal states** (goal with +1, pit with −1). Each step costs a small penalty to encourage shorter paths. Optional **slip** probability executes a random action instead of the intended one.

In [ ]:
class Gridworld:
    def __init__(self, H=5, W=7, step_cost=-0.02, slip=0.0):
        self.H, self.W = H, W
        self.step_cost = step_cost
        self.slip = slip
        self.walls = set([(1,1), (1,5), (2,3), (3,3)])  # (row, col)
        self.terminals = {(0,6): +1.0, (4,0): -1.0}
        self.reset()

    def reset(self, start=(4,6)):
        self.s = start
        return self.s

    def is_terminal(self, s):
        return s in self.terminals

    def in_bounds(self, r, c):
        return 0 <= r < self.H and 0 <= c < self.W

    def step(self, a_idx):
        if self.is_terminal(self.s):
            return self.s, 0.0, True
        # slipping
        if rng.random() < self.slip:
            a_idx = int(rng.integers(0, 4))
        dr, dc = ACTIONS[a_idx]
        r, c = self.s
        nr, nc = r+dr, c+dc
        # bounce on wall or out of bounds
        if not self.in_bounds(nr, nc) or (nr, nc) in self.walls:
            nr, nc = r, c
        s2 = (nr, nc)
        rwd = self.step_cost
        done = False
        if s2 in self.terminals:
            rwd = self.terminals[s2]
            done = True
        self.s = s2
        return s2, rwd, done

    # Full model for DP
    def states(self):
        S = []
        for r in range(self.H):
            for c in range(self.W):
                if (r,c) not in self.walls:
                    S.append((r,c))
        return S

    def model(self, s, a_idx):
        # returns list of (prob, s', reward)
        if s in self.terminals:
            return [(1.0, s, 0.0)]
        outcomes = []
        for a2 in range(4):
            prob = (1.0 - self.slip) if a2 == a_idx else (self.slip/3.0)
            dr, dc = ACTIONS[a2]
            r, c = s
            nr, nc = r+dr, c+dc
            if not self.in_bounds(nr, nc) or (nr, nc) in self.walls:
                nr, nc = r, c
            s2 = (nr, nc)
            rwd = self.step_cost
            if s2 in self.terminals:
                rwd = self.terminals[s2]
            outcomes.append((prob, s2, rwd))
        return outcomes

## 3) Value Iteration (Dynamic Programming)

We compute $V^*$ by iterating the **Bellman optimality** operator:
$$V_{k+1}(s) \leftarrow \max_a \sum_{s'} P(s'\mid s,a)\,[R(s,a,s') + \gamma V_k(s')].$$
Then derive the greedy policy $\pi^*(s) = \arg\max_a Q^*(s,a)$.

In [ ]:
def value_iteration(env, gamma=0.98, tol=1e-6, max_iter=10_000):
    S = env.states()
    idx = {s:i for i,s in enumerate(S)}
    V = np.zeros(len(S))
    for it in range(max_iter):
        dV = 0.0
        V_new = V.copy()
        for si, s in enumerate(S):
            if env.is_terminal(s):
                V_new[si] = 0.0
                continue
            q_vals = []
            for a in range(4):
                q = 0.0
                for p, sp, r in env.model(s, a):
                    q += p*(r + gamma*V[idx[sp]])
                q_vals.append(q)
            V_new[si] = np.max(q_vals)
            dV = max(dV, abs(V_new[si]-V[si]))
        V = V_new
        if dV < tol:
            break
    # Derive greedy policy
    Pi = np.zeros(len(S), dtype=int)
    Q = np.zeros((len(S), 4))
    for si, s in enumerate(S):
        if env.is_terminal(s):
            Pi[si] = 0; Q[si,:] = 0.0
            continue
        for a in range(4):
            q = 0.0
            for p, sp, r in env.model(s, a):
                q += p*(r + gamma*V[idx[sp]])
            Q[si,a] = q
        Pi[si] = int(np.argmax(Q[si]))
    return S, idx, V, Q, Pi

## 4) Q‑Learning (Model‑Free)

Update rule with learning rate $\alpha$ and discount $\gamma$:
$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[r + \gamma\max_{a'} Q(s',a') - Q(s,a)].$$
Exploration via **ε‑greedy** with optional decay $\epsilon_t = \max(\epsilon_{min}, \epsilon_0\cdot\text{decay}^t)$.

In [ ]:
def q_learning(env, episodes=5_000, alpha=0.2, gamma=0.98, eps0=0.2, eps_min=0.02, eps_decay=0.999):
    Q = {(r,c): np.zeros(4) for r in range(env.H) for c in range(env.W) if (r,c) not in env.walls}
    returns = np.zeros(episodes)
    for ep in range(episodes):
        s = env.reset()
        done = False
        eps = max(eps_min, eps0*(eps_decay**ep))
        G = 0.0
        steps = 0
        while not done and steps < 500:
            if env.is_terminal(s):
                break
            if rng.random() < eps:
                a = int(rng.integers(0, 4))
            else:
                a = int(np.argmax(Q[s]))
            s2, r, done = env.step(a)
            if s2 in Q:
                td_target = r + gamma*np.max(Q[s2])
            else:
                td_target = r
            td_err = td_target - Q[s][a]
            Q[s][a] += alpha*td_err
            s = s2
            G += r
            steps += 1
        returns[ep] = G
    return Q, returns

## 5) Visualization Helpers

In [ ]:
def grid_from_state_values(env, S, V, fill=np.nan):
    G = np.full((env.H, env.W), fill, dtype=float)
    for s, v in zip(S, V):
        G[s] = v
    for w in env.walls: G[w] = np.nan
    for t, r in env.terminals.items(): G[t] = r
    return G

def grid_from_policy(env, S, Pi, fill=' '):
    G = np.full((env.H, env.W), fill, dtype=object)
    for s, a in zip(S, Pi):
        G[s] = A_NAMES[a]
    for w in env.walls: G[w] = '█'
    for t, r in env.terminals.items(): G[t] = f"{r:+.0f}"
    return G

def plot_heatmap(ax, M, title, cmap='viridis'):
    im = ax.imshow(M, interpolation='nearest', cmap=cmap)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    for (i,j), val in np.ndenumerate(M):
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8, color='w')
    return im

def plot_policy(ax, env, arrows):
    ax.imshow(np.ones((env.H, env.W))*0.95, interpolation='nearest', cmap='gray')
    for r in range(env.H):
        for c in range(env.W):
            if (r,c) in env.walls:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='black'))
            elif (r,c) in env.terminals:
                col = 'green' if env.terminals[(r,c)]>0 else 'red'
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color=col, alpha=0.6))
            else:
                ax.text(c, r, arrows[r,c], ha='center', va='center', fontsize=14)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title("Policy (arrows)")

## 6) Solve with Value Iteration and Q‑Learning

In [ ]:
env = Gridworld(H=5, W=7, step_cost=-0.02, slip=0.1)

# Value Iteration
S, idx, V_star, Q_star, Pi_star = value_iteration(env, gamma=0.98, tol=1e-7, max_iter=10_000)

# Q-Learning
Q_hat, returns = q_learning(env, episodes=6000, alpha=0.25, gamma=0.98, eps0=0.3, eps_min=0.02, eps_decay=0.999)

# Build V_hat and Pi_hat from learned Q
V_hat = np.zeros_like(V_star)
Pi_hat = np.zeros_like(Pi_star)
for s, i in idx.items():
    V_hat[i] = np.max(Q_hat[s])
    Pi_hat[i] = int(np.argmax(Q_hat[s]))

## 7) Compare Values and Policies

In [ ]:
V_grid_star = grid_from_state_values(env, S, V_star)
V_grid_hat  = grid_from_state_values(env, S, V_hat)

fig = plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1); plot_heatmap(ax1, V_grid_star, "V* (Value Iteration)")
ax2 = plt.subplot(1,2,2); plot_heatmap(ax2, V_grid_hat,  "V̂ (Q‑Learning)")
plt.tight_layout(); plt.show()

# Policies
Pi_grid_star = grid_from_policy(env, S, Pi_star)
Pi_grid_hat  = grid_from_policy(env, S, Pi_hat)

fig = plt.figure(figsize=(10,4))
ax1 = plt.subplot(1,2,1); plot_policy(ax1, env, Pi_grid_star)
ax1.set_title("π* (from VI)")
ax2 = plt.subplot(1,2,2); plot_policy(ax2, env, Pi_grid_hat)
ax2.set_title("π̂ (from QL)")
plt.tight_layout(); plt.show()

## 8) Learning Curve (Return per Episode)

In [ ]:
fig = plt.figure(figsize=(7,4))
plt.plot(returns, alpha=0.5, label="episode return")
# simple moving average
w = 100
ma = np.convolve(returns, np.ones(w)/w, mode='same')
plt.plot(ma, label=f"moving avg (w={w})")
plt.xlabel("Episode"); plt.ylabel("Return"); plt.title("Q‑Learning Learning Curve"); plt.legend()
plt.tight_layout(); plt.show()

## 9) Trajectory Rollout under Learned Policy

In [ ]:
def act_greedy(Q, s):
    return int(np.argmax(Q[s])) if s in Q else 0

def rollout(env, Q, start=(4,6), max_steps=200):
    s = env.reset(start=start)
    traj = [s]
    ret = 0.0; done=False
    for t in range(max_steps):
        if env.is_terminal(s): break
        a = act_greedy(Q, s)
        s, r, done = env.step(a)
        traj.append(s); ret += r
        if done: break
    return traj, ret

traj, ret = rollout(env, Q_hat, start=(4,6))
print("Rollout return (learned policy):", ret)

# Plot path
fig = plt.figure(figsize=(5,4.5))
ax = plt.gca()
ax.imshow(np.ones((env.H, env.W))*0.95, interpolation='nearest', cmap='gray')
for wcell in env.walls:
    ax.add_patch(plt.Rectangle((wcell[1]-0.5, wcell[0]-0.5), 1, 1, color='black'))
for tcell, rwd in env.terminals.items():
    col = 'green' if rwd>0 else 'red'
    ax.add_patch(plt.Rectangle((tcell[1]-0.5, tcell[0]-0.5), 1, 1, color=col, alpha=0.6))
xs = [c for (r,c) in traj]; ys = [r for (r,c) in traj]
ax.plot(xs, ys, marker='o')
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("Trajectory under learned policy")
plt.tight_layout(); plt.show()

## 10) Save Artifacts & Download

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

art = {
    "H": env.H, "W": env.W, "walls": list(map(list, env.walls)),
    "terminals": {str(k): v for k,v in env.terminals.items()},
    "step_cost": env.step_cost, "slip": env.slip
}
np.savez("artifacts/gridworld_results.npz",
         S=np.array(S, dtype=object),
         V_star=V_star, Pi_star=Pi_star,
         V_hat=V_hat, Pi_hat=Pi_hat,
         returns=returns)
with open("artifacts/gridworld_meta.json", "w") as f:
    json.dump(art, f, indent=2)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 11) Exercises & Extensions

- Replace ε‑greedy with **Boltzmann (softmax)** exploration.
- Make the environment **non‑stationary** (move terminals) and add **decaying** or **adaptive α**.
- Implement **SARSA** and compare to Q‑learning under the same exploration schedule.
- Add **stochastic slip** in VI and verify policy robustness vs. model‑free learning.
- Extend to **stochastic rewards** or multiple goals with different payoffs.